In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [2]:
import yaml

with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

Opening both files using "yaml"

In [3]:
from pathlib import Path
import pandas as pd

base_path = Path("../config.yaml").parent
red_wines = pd.read_csv(base_path / config["input_data"]["file1"])
red_wines.head()

,Name,Country,Region,Winery,Rating,NumberOfRatings,Price,Year
0,Pomerol 2011,France,Pomerol,Château La Providence,4.2,100,95.00,2011
1,Lirac 2017,France,Lirac,Château Mont-Redon,4.3,100,15.50,2017
2,Erta e China Rosso di Toscana 2015,Italy,Toscana,Renzo Masi,3.9,100,7.45,2015
3,Bardolino 2019,Italy,Bardolino,Cavalchina,3.5,100,8.72,2019
4,Ried Scheibner Pinot Noir 2016,Austria,Carnuntum,Markowitsch,3.9,100,29.15,2016


In [4]:
red_wines.info()

<class 'pandas.DataFrame'>
RangeIndex: 8666 entries, 0 to 8665
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             8666 non-null   str    
 1   Country          8666 non-null   str    
 2   Region           8666 non-null   str    
 3   Winery           8666 non-null   str    
 4   Rating           8666 non-null   float64
 5   NumberOfRatings  8666 non-null   int64  
 6   Price            8666 non-null   float64
 7   Year             8666 non-null   str    
dtypes: float64(2), int64(1), str(5)
memory usage: 541.8 KB


In [5]:
wine_varieties = pd.read_csv(base_path / config["input_data"]["file2"])
wine_varieties.head()

,Variety
0,Abouriou
1,Abrustine
2,Absinthe
3,Acadie Blanc
4,Acolon


Extracting Variety out of Wine Names so it can be merged with Varieties.csv

In [6]:
common_varieties = wine_varieties["Variety"].unique()

def find_variety(name):
    for v in common_varieties:
        if v in name:
            return v
    return "Other"

red_wines["variety"] = red_wines["Name"].apply(find_variety)
red_wines.head()

,Name,Country,Region,Winery,Rating,NumberOfRatings,Price,Year,variety
0,Pomerol 2011,France,Pomerol,Château La Providence,4.2,100,95.00,2011,Other
1,Lirac 2017,France,Lirac,Château Mont-Redon,4.3,100,15.50,2017,Other
2,Erta e China Rosso di Toscana 2015,Italy,Toscana,Renzo Masi,3.9,100,7.45,2015,Other
3,Bardolino 2019,Italy,Bardolino,Cavalchina,3.5,100,8.72,2019,Other
4,Ried Scheibner Pinot Noir 2016,Austria,Carnuntum,Markowitsch,3.9,100,29.15,2016,Pinot Noir


Checking for null values

In [7]:
print(red_wines.isnull().sum())
print(wine_varieties.isnull().sum())
print(red_wines.isnull().mean() * 100)  # as percentages
print(wine_varieties.isnull().mean() * 100)  # as percentages

Name               0
Country            0
Region             0
Winery             0
Rating             0
NumberOfRatings    0
Price              0
Year               0
variety            0
dtype: int64
Variety    0
dtype: int64
Name               0.0
Country            0.0
Region             0.0
Winery             0.0
Rating             0.0
NumberOfRatings    0.0
Price              0.0
Year               0.0
variety            0.0
dtype: float64
Variety    0.0
dtype: float64


Counting Varieties in red_wines and understanding how many "others" there are. Merging might not be a good decision and "varieties" not needed.

In [8]:
red_wines["variety"].value_counts()

variety
Other                 5550
Cabernet Sauvignon     549
Merlot                 269
Pinot Noir             260
Shiraz                 231
                      ... 
Carmenere                1
Monica                   1
Giro                     1
Brachetto                1
Albarossa                1
Name: count, Length: 105, dtype: int64

I standardized all category labels to ensure consistency and readability, then exported a clean ML-ready dataset with the final variety_ml feature.

In [9]:
import pandas as pd
import unicodedata
import re

def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

BASE_PATH = "../data/raw/"

red = pd.read_csv(BASE_PATH + "red.csv")
varieties = pd.read_csv(BASE_PATH + "varieties.csv")

red.columns = red.columns.str.strip().str.lower()
varieties.columns = varieties.columns.str.strip().str.lower()

red["name_clean"] = red["name"].map(normalize)
red["region_clean"] = red["region"].map(normalize)
varieties["variety_clean"] = varieties["variety"].map(normalize)

valid_varieties = sorted(
    [v for v in varieties["variety_clean"].dropna().unique() if v and len(v) > 2],
    key=len,
    reverse=True
)

variety_pattern = re.compile(
    r"\b(" + "|".join(re.escape(v) for v in valid_varieties) + r")\b"
)

REGION_MAP = {
    "pomerol": "Bordeaux Blend",
    "margaux": "Bordeaux Blend",
    "pauillac": "Bordeaux Blend",
    "saint emilion": "Bordeaux Blend",
    "haut medoc": "Bordeaux Blend",
    "medoc": "Bordeaux Blend",
    "saint julien": "Bordeaux Blend",
    "saint estephe": "Bordeaux Blend",
    "pessac leognan": "Bordeaux Blend",
    "fronsac": "Bordeaux Blend",
    "graves": "Bordeaux Blend",

    "gevrey chambertin": "Pinot Noir",
    "chambolle musigny": "Pinot Noir",
    "pommard": "Pinot Noir",
    "nuits saint georges": "Pinot Noir",
    "vosne romanee": "Pinot Noir",
    "cote de beaune": "Pinot Noir",

    "morgon": "Gamay",
    "fleurie": "Gamay",
    "beaujolais": "Gamay",

    "lirac": "Rhone Blend",
    "gigondas": "Rhone Blend",
    "bandol": "Rhone Blend",
    "ventoux": "Rhone Blend",
    "corbieres": "Rhone Blend",
    "larzac": "Rhone Blend",
    "chateauneuf du pape": "Rhone Blend",
    "cotes du rhone": "Rhone Blend",
    "gsm": "Rhone Blend",
    "saint joseph": "Syrah",
    "crozes hermitage": "Syrah",

    "chianti": "Sangiovese",
    "chianti classico": "Sangiovese",
    "brunello": "Sangiovese",
    "morellino di scansano": "Sangiovese",
    "rosso di toscana": "Sangiovese",
    "toscana": "Generic Red",
    "barolo": "Nebbiolo",
    "barbaresco": "Nebbiolo",
    "amarone": "Corvina",
    "valpolicella": "Corvina",
    "bardolino": "Corvina",
    "appassimento": "Corvina",
    "salice salentino": "Negroamaro",
    "cannonau di sardegna": "Grenache",
    "pinot nero": "Pinot Noir",

    "rioja": "Tempranillo",
    "ribera del duero": "Tempranillo",
    "crianza": "Tempranillo",
    "reserva": "Tempranillo",
    "roble": "Tempranillo",
    "tinto": "Tempranillo",
    "garnacha": "Grenache",

    "spatburgunder": "Pinot Noir",
    "madiran": "Tannat",
    "cahors": "Malbec",
    "monastrell": "Mourvedre",
    "pannobile": "Blend",
    "rouge": "Generic Red",
}

def map_from_text(text):
    if not text:
        return None

    match = variety_pattern.search(text)
    if match:
        return match.group(1).title()

    for key, value in REGION_MAP.items():
        if key in text:
            return value

    if "blend" in text:
        return "Blend"
    if "bordeaux" in text:
        return "Bordeaux Blend"
    if "rhone" in text:
        return "Rhone Blend"
    if "rosso" in text or "rouge" in text or "red" in text:
        return "Generic Red"

    return None

def infer_variety(row):
    region_hit = map_from_text(row["region_clean"])
    if region_hit:
        return region_hit

    name_hit = map_from_text(row["name_clean"])
    if name_hit:
        return name_hit

    return "Unknown"

red["variety"] = red.apply(infer_variety, axis=1)

# collapse similar labels
red["variety"] = red["variety"].replace({
    "Brunello": "Sangiovese",
    "Cabernet Sauvignon Merlot": "Bordeaux Blend"
})

# keep only common ML classes, group the rest
TOP_CLASSES = {
    "Bordeaux Blend",
    "Tempranillo",
    "Cabernet Sauvignon",
    "Generic Red",
    "Sangiovese",
    "Pinot Noir",
    "Nebbiolo",
    "Rhone Blend",
    "Merlot",
    "Corvina",
    "Shiraz",
    "Syrah",
    "Malbec",
    "Primitivo",
    "Barbera",
    "Montepulciano",
    "Grenache",
    "Blend",
    "Unknown"
}

red["variety_ml"] = red["variety"].apply(
    lambda x: x if x in TOP_CLASSES else "Other Red"
)

# show final ML feature distribution
ml_counts = red["variety_ml"].value_counts().reset_index()
ml_counts.columns = ["variety_ml", "count"]

print("ML-ready variety feature counts:")
print(ml_counts)

print("\nPreview:")
print(red[["name", "region", "variety", "variety_ml"]].head(20))

ML-ready variety feature counts:
            variety_ml  count
0              Unknown   1390
1       Bordeaux Blend   1216
2            Other Red    889
3          Tempranillo    841
4          Generic Red    732
5           Pinot Noir    479
6   Cabernet Sauvignon    478
7           Sangiovese    447
8          Rhone Blend    304
9             Nebbiolo    271
10              Merlot    259
11             Corvina    240
12              Shiraz    219
13               Syrah    206
14              Malbec    178
15           Primitivo    129
16             Barbera    127
17       Montepulciano    113
18            Grenache     75
19               Blend     73

Preview:
                                                name                   region  \
0                                       Pomerol 2011                  Pomerol   
1                                         Lirac 2017                    Lirac   
2                 Erta e China Rosso di Toscana 2015                  Toscana   
3  

In [10]:
red["variety_ml"]

0           Bordeaux Blend
1              Rhone Blend
2              Generic Red
3                  Corvina
4               Pinot Noir
               ...        
8661                 Syrah
8662           Generic Red
8663        Bordeaux Blend
8664                Shiraz
8665    Cabernet Sauvignon
Name: variety_ml, Length: 8666, dtype: str

In [11]:
print(red["variety_ml"].value_counts())

variety_ml
Unknown               1390
Bordeaux Blend        1216
Other Red              889
Tempranillo            841
Generic Red            732
Pinot Noir             479
Cabernet Sauvignon     478
Sangiovese             447
Rhone Blend            304
Nebbiolo               271
Merlot                 259
Corvina                240
Shiraz                 219
Syrah                  206
Malbec                 178
Primitivo              129
Barbera                127
Montepulciano          113
Grenache                75
Blend                   73
Name: count, dtype: int64


In [12]:
ml_counts = red["variety_ml"].value_counts().reset_index()
ml_counts.columns = ["variety_ml", "count"]
print(ml_counts)

            variety_ml  count
0              Unknown   1390
1       Bordeaux Blend   1216
2            Other Red    889
3          Tempranillo    841
4          Generic Red    732
5           Pinot Noir    479
6   Cabernet Sauvignon    478
7           Sangiovese    447
8          Rhone Blend    304
9             Nebbiolo    271
10              Merlot    259
11             Corvina    240
12              Shiraz    219
13               Syrah    206
14              Malbec    178
15           Primitivo    129
16             Barbera    127
17       Montepulciano    113
18            Grenache     75
19               Blend     73


In [13]:
print(red[["name", "region", "variety", "variety_ml"]].head(20))

                                                name                   region  \
0                                       Pomerol 2011                  Pomerol   
1                                         Lirac 2017                    Lirac   
2                 Erta e China Rosso di Toscana 2015                  Toscana   
3                                     Bardolino 2019                Bardolino   
4                     Ried Scheibner Pinot Noir 2016                Carnuntum   
5                   Gigondas (Nobles Terrasses) 2017                 Gigondas   
6                  Marion's Vineyard Pinot Noir 2016                Wairarapa   
7                                     Red Blend 2014             Itata Valley   
8                                       Chianti 2015                  Chianti   
9                                     Tradition 2014                Minervois   
10                              Chianti Riserva 2013                  Chianti   
11                          

In [14]:
# red.to_csv("../data/raw/red_ml_ready.csv", index=False)

In [15]:
print(red["variety_ml"].unique())

<StringArray>
[    'Bordeaux Blend',        'Rhone Blend',        'Generic Red',
            'Corvina',         'Pinot Noir',              'Blend',
         'Sangiovese',            'Unknown',          'Primitivo',
             'Shiraz', 'Cabernet Sauvignon',          'Other Red',
        'Tempranillo',            'Barbera',              'Syrah',
           'Grenache',             'Merlot',           'Nebbiolo',
             'Malbec',      'Montepulciano']
Length: 20, dtype: str


In [16]:
red["variety_ml"] = red["variety_ml"].replace({
    "Other Red": "Rare Varieties",
    "Generic Red": "Unspecified Red",
    "Unknown": "Unknown Variety"
})

In [17]:
print(red["variety_ml"].value_counts())

variety_ml
Unknown Variety       1390
Bordeaux Blend        1216
Rare Varieties         889
Tempranillo            841
Unspecified Red        732
Pinot Noir             479
Cabernet Sauvignon     478
Sangiovese             447
Rhone Blend            304
Nebbiolo               271
Merlot                 259
Corvina                240
Shiraz                 219
Syrah                  206
Malbec                 178
Primitivo              129
Barbera                127
Montepulciano          113
Grenache                75
Blend                   73
Name: count, dtype: int64


In [18]:
RENAME_MAP = {
    "Other Red": "Rare Varieties",
    "Generic Red": "Unspecified Red",
    "Unknown": "Unknown Variety",

    # Optional consistency cleanup
    "Bordeaux Blend": "Bordeaux Blend",
    "Rhone Blend": "Rhône Blend",
    "Cabernet Sauvignon": "Cabernet Sauvignon",
    "Pinot Noir": "Pinot Noir",
    "Merlot": "Merlot",
    "Tempranillo": "Tempranillo",
    "Sangiovese": "Sangiovese",
    "Nebbiolo": "Nebbiolo",
    "Syrah": "Syrah",
    "Shiraz": "Shiraz",
    "Malbec": "Malbec",
    "Grenache": "Grenache",
    "Corvina": "Corvina",
    "Barbera": "Barbera",
    "Montepulciano": "Montepulciano",
    "Zinfandel": "Zinfandel",
    "Carmenere": "Carménère",
    "Nero D Avola": "Nero d'Avola"
}

red["variety_ml"] = red["variety_ml"].replace(RENAME_MAP)

In [19]:
print(red["variety_ml"].value_counts())

variety_ml
Unknown Variety       1390
Bordeaux Blend        1216
Rare Varieties         889
Tempranillo            841
Unspecified Red        732
Pinot Noir             479
Cabernet Sauvignon     478
Sangiovese             447
Rhône Blend            304
Nebbiolo               271
Merlot                 259
Corvina                240
Shiraz                 219
Syrah                  206
Malbec                 178
Primitivo              129
Barbera                127
Montepulciano          113
Grenache                75
Blend                   73
Name: count, dtype: int64


In [20]:
print(red[["name", "region", "variety_ml"]].head(20))

                                                name                   region  \
0                                       Pomerol 2011                  Pomerol   
1                                         Lirac 2017                    Lirac   
2                 Erta e China Rosso di Toscana 2015                  Toscana   
3                                     Bardolino 2019                Bardolino   
4                     Ried Scheibner Pinot Noir 2016                Carnuntum   
5                   Gigondas (Nobles Terrasses) 2017                 Gigondas   
6                  Marion's Vineyard Pinot Noir 2016                Wairarapa   
7                                     Red Blend 2014             Itata Valley   
8                                       Chianti 2015                  Chianti   
9                                     Tradition 2014                Minervois   
10                              Chianti Riserva 2013                  Chianti   
11                          

In [21]:
red.to_csv("../data/raw/red_ml_final.csv", index=False)

In [22]:
# Minimal dataset for modeling only
ml_df = red[[
    "country",
    "region",
    "price",
    "rating",
    "variety_ml"
]]

ml_df.to_csv("../data/raw/red_ml_model_input.csv", index=False)

In [23]:
print(red["variety_ml"].unique())

<StringArray>
[    'Bordeaux Blend',        'Rhône Blend',    'Unspecified Red',
            'Corvina',         'Pinot Noir',              'Blend',
         'Sangiovese',    'Unknown Variety',          'Primitivo',
             'Shiraz', 'Cabernet Sauvignon',     'Rare Varieties',
        'Tempranillo',            'Barbera',              'Syrah',
           'Grenache',             'Merlot',           'Nebbiolo',
             'Malbec',      'Montepulciano']
Length: 20, dtype: str


In [ ]:
ml_df = pd.read_csv("../data/raw/red_ml_final.csv")
ml_df.head()
ml_df.info()


SyntaxError: invalid syntax (1445453478.py, line 2)